In [34]:
from pathlib import Path

import geomeffibem
import openstudio
import numpy as np

openstudio.Logger.instance().standardOutLogger().setLogLevel(openstudio.Warn)

In [72]:
def create_opaque_construction(m):
    """Create an opaque construction with a 200mm concrete layer and R13 insulation."""
    c = openstudio.model.Construction(m)
    c.setName("Opaque Construction")
    m15_200mm_heavyweight_concrete = openstudio.model.StandardOpaqueMaterial(m)
    m15_200mm_heavyweight_concrete.setName("M15 200mm heavyweight concrete")
    m15_200mm_heavyweight_concrete.setRoughness("MediumRough")
    m15_200mm_heavyweight_concrete.setThickness(0.2032)
    m15_200mm_heavyweight_concrete.setThermalConductivity(1.95)
    m15_200mm_heavyweight_concrete.setDensity(2240.0)
    m15_200mm_heavyweight_concrete.setSpecificHeat(900.0)
    insulation_mat = openstudio.model.MasslessOpaqueMaterial(m)
    insulation_mat.setName("R13-IP")
    insulation_mat.setThermalResistance(openstudio.convert(13, "ft^2*h*R/Btu", "m^2*K/W").get())
    assert c.setLayers([m15_200mm_heavyweight_concrete, insulation_mat])
    assert len(c.layers()) == 2
    return c


def create_window_construction(m):
    """Create a window construction with simple glazing."""
    simple_glazing = openstudio.model.SimpleGlazing(m)
    simple_glazing.setName("Simple Glazing Mat")
    simple_glazing.setSolarHeatGainCoefficient(0.65)
    window_cons = openstudio.model.Construction(m)
    window_cons.setName("Simple Glazing")
    window_cons.setLayers([simple_glazing])

    return window_cons


def create_space(
    name: str,
    m: openstudio.model.Model,
    xOffset: int = 0,
    zOffset: int = 0,
    floor_width: float = 10.0,
    floor_depth: float = 10.0,
    floor_height: float = 3.0,
):
    """Create a space."""
    floor_sf = geomeffibem.Surface.Floor(min_x=0.0, max_x=floor_width, min_y=0.0, max_y=floor_depth, z=0.0)

    space = openstudio.model.Space.fromFloorPrint(floor_sf.to_Point3dVector(), floor_height, m, name).get()
    space.setXOrigin(floor_width * xOffset)
    space.setZOrigin(floor_height * zOffset)

    z = openstudio.model.ThermalZone(m)
    z.setName(name.replace("Space", "Zone"))
    space.setThermalZone(z)
    return space


def add_interior_partition(space2):
    """Add a desk to space 2."""
    deskGroup = openstudio.model.InteriorPartitionSurfaceGroup(space2.model())
    deskGroup.setSpace(space2)

    deskPoints = [
        openstudio.Point3d(5, 8, 1),
        openstudio.Point3d(5, 6, 1),
        openstudio.Point3d(8, 6, 1),
        openstudio.Point3d(8, 8, 1),
    ]
    desk = openstudio.model.InteriorPartitionSurface(deskPoints, space2.model())
    desk.setInteriorPartitionSurfaceGroup(deskGroup)
    return desk


def add_skylight(space) -> openstudio.model.SubSurface:
    roof = next(s for s in space.surfaces() if s.surfaceType().lower() == "roofceiling")

    vertices = roof.vertices()
    g = openstudio.getCentroid(vertices).get()
    scale_factor = 0.4**0.5  # sqrt(40%)

    new_vertices = []
    for vertex in vertices:
        # Point3d - Point3d = Vector3d
        # Vector from centroid to vertex (GA, GB, GC, etc)
        centroid_vector = vertex - g

        # Resize the vector (done in place) according to scale_factor
        centroid_vector.setLength(centroid_vector.length() * scale_factor)

        # Move the vertex toward the centroid
        vertex = g + centroid_vector

        new_vertices.append(vertex)
    ss = openstudio.model.SubSurface(new_vertices, m)
    ss.setName("ExteriorWindow - Skylight")
    ss.setSurface(roof)

    return ss


def add_door(wall, subsurface_type: str, width=0.8, height=2.0) -> geomeffibem.Surface:
    """Create a door (default 80cm x 200cm) centered."""
    surfaceVertices = wall.vertices()

    # new coordinate system has z' in direction of outward normal, y' is up
    transformation = openstudio.Transformation.alignFace(surfaceVertices)
    faceVertices = transformation.inverse() * surfaceVertices

    xs = [pt.x() for pt in faceVertices]
    min_x = min(xs)
    max_x = max(xs)

    ys = [pt.y() for pt in faceVertices]
    min_y = min(ys)
    # max_y = max(ys)

    center_x = min_x + (max_x - min_x) / 2

    viewMinX = center_x - width / 2.0
    viewMaxX = center_x + width / 2.0

    viewVertices = [
        openstudio.Point3d(viewMinX, min_y + height, 0),
        openstudio.Point3d(viewMinX, min_y, 0),
        openstudio.Point3d(viewMaxX, min_y, 0),
        openstudio.Point3d(viewMaxX, min_y + height, 0),
    ]
    # Go back to input coordinate system
    viewVertices = transformation * viewVertices
    ss = openstudio.model.SubSurface(viewVertices, wall.model())
    win_type = "Exterior" if wall.outsideBoundaryCondition().lower() == "outdoors" else "Interior"
    ss.setName(f"{win_type}Door - {subsurface_type}")
    ss.setSurface(wall)
    ss.setSubSurfaceType(subsurface_type)
    return ss


def add_window(wall, subsurface_type: str):
    """Create a window on the wall."""
    ss = wall.setWindowToWallRatio(0.4).get()
    win_type = "Exterior" if wall.outsideBoundaryCondition().lower() == "outdoors" else "Interior"
    ss.setName(f"{win_type}Window - {subsurface_type}")
    ss.setSubSurfaceType(subsurface_type)
    return ss


def get_construction_name(surface_boundary_type: str, surface_type: str):
    return f"{surface_boundary_type} {surface_type} Construction"


def get_constructions_and_materials():
    idd = openstudio.IddFile.load(
        Path("/home/julien/Software/Others/EnergyPlus-build-release/Products/Energy+.idd")
    ).get()

    o_ = idd.getObject("DefaultConstructionSet")
    assert o_.is_initialized()
    idd_default_construction_set = o_.get()

    o_ = idd.getObject("DefaultSurfaceConstructions")
    assert o_.is_initialized()
    idd_default_surfaces_construction = o_.get()

    o_ = idd.getObject("DefaultSubSurfaceConstructions")
    assert o_.is_initialized()
    idd_default_subsurfaces_construction = o_.get()

    objects = []
    # name: is_glazed
    construction_infos = {}
    mat_glazing_name = "Mat Glazing"
    mat_opaque_name = "Mat Opaque"

    dc = openstudio.IdfObject(idd_default_construction_set)
    dc.setName("Default Construction Set")

    objects.append(dc)

    # Surface Constructions
    surface_types = ["Floor", "Wall", "Roof"]

    surface_boundary_types = ["Exterior", "Interior", "Ground"]

    for k, surface_boundary_type in enumerate(surface_boundary_types):

        sc = openstudio.IdfObject(idd_default_surfaces_construction)
        name = f"{surface_boundary_type} Surface Constructions"
        sc.setName(name)
        dc.setString(k + 1, name)
        for i, surface_type in enumerate(surface_types):
            cons_name = get_construction_name(surface_boundary_type=surface_boundary_type, surface_type=surface_type)
            construction_infos[cons_name] = False
            sc.setString(i + 1, cons_name)

        objects.append(sc)

    # SubSurface Constructions
    subsurface_types = [
        "FixedWindow",
        "OperableWindow",
        "Door",
        "GlassDoor",
        "OverheadDoor",
        "Skylight",
        "TubularDaylightDome",
        "TubularDaylightDiffuser",
    ]

    subsurface_boundary_types = ["Exterior", "Interior"]

    for k, subsurface_boundary_type in enumerate(subsurface_boundary_types):

        sc = openstudio.IdfObject(idd_default_subsurfaces_construction)
        name = f"{subsurface_boundary_type} SubSurface Constructions"
        sc.setName(name)
        dc.setString(k + 4, name)
        for i, subsurface_type in enumerate(subsurface_types):
            cons_name = get_construction_name(
                surface_boundary_type=subsurface_boundary_type, surface_type=subsurface_type
            )
            construction_infos[cons_name] = subsurface_type not in ["Door", "OverheadDoor"]
            sc.setString(i + 1, cons_name)

        objects.append(sc)

    # Interior Partition
    name = "Interior Partition Construction"
    dc.setString(6, name)
    construction_infos[name] = False

    # Adiabatic Partition
    name = "Adiabatic Surface Construction"
    dc.setString(7, name)
    construction_infos[name] = False

    for construction_name, is_glazed in construction_infos.items():
        construction = openstudio.IdfObject(idd.getObject("Construction").get())
        construction.setName(construction_name)
        if is_glazed:
            construction.setString(1, mat_glazing_name)
        else:
            construction.setString(1, mat_opaque_name)
        objects.append(construction)

    mat_opaque = openstudio.IdfObject(idd.getObject("Material:NoMass").get())
    mat_opaque.setName(mat_opaque_name)
    mat_opaque.setString(1, "Rough")
    mat_opaque.setDouble(2, 0.5)
    mat_opaque.setDouble(3, 0.9)
    mat_opaque.setDouble(4, 0.9)
    mat_opaque.setDouble(5, 0.9)
    objects.append(mat_opaque)

    mat_glazing = openstudio.IdfObject(idd.getObject("WindowMaterial:SimpleGlazingSystem").get())
    mat_glazing.setName(mat_glazing_name)
    mat_glazing.setDouble(1, 0.1)
    mat_glazing.setDouble(2, 0.65)
    objects.append(mat_glazing)

    return objects


In [38]:
m = openstudio.model.Model()
c_opaque = create_opaque_construction(m)
c_window = create_window_construction(m)
space1 = create_space(name="Space1", m=m, xOffset=0, zOffset=0)
space2 = create_space(name="Space2", m=m, xOffset=1, zOffset=0)
space3 = create_space(name="Space3", m=m, xOffset=0, zOffset=1)
space4 = create_space(name="Space4", m=m, xOffset=1, zOffset=1)

[s.setConstruction(c_opaque) for s in m.getSurfaces()]

openstudio.model.matchSurfaces(openstudio.model.SpaceVector([space1, space2, space3, space4]))

desk = add_interior_partition(space2=space2)

space2_ext_walls = [
    s
    for s in space2.surfaces()
    if s.surfaceType().lower() == "wall" and s.outsideBoundaryCondition().lower() == "outdoors"
]
assert len(space2_ext_walls) == 3, f"Expected 3 exterior walls for space 2, found {len(space2_ext_walls)}"

space2_ext_walls[0].setOutsideBoundaryCondition("Adiabatic")
space2_ext_walls[1].setOutsideBoundaryCondition("Ground")

ext_walls = sorted(
    [
        s
        for s in m.getSurfaces()
        if s.surfaceType().lower() == "wall" and s.outsideBoundaryCondition().lower() == "outdoors"
    ],
    key=lambda s: s.nameString(),
)
assert len(ext_walls) == 10  # 12 total walls - 2 I reassigned = 10 exterior walls
ss = add_door(ext_walls[0], "Door")
ss.setConstruction(c_opaque)
ss = add_door(ext_walls[1], "OverheadDoor", width=5.0, height=2.5)
ss.setConstruction(c_opaque)
ss = add_door(ext_walls[2], "GlassDoor")
ss.setConstruction(c_window)

ss = add_skylight(space4)
ss.setConstruction(c_window)

for w, ss_type in zip(ext_walls[3:], ["FixedWindow", "OperableWindow", "Window"]):
    ss = add_window(w, subsurface_type=ss_type)
    ss.setConstruction(c_window)

int_walls = sorted(
    [
        s
        for s in m.getSurfaces()
        if s.surfaceType().lower() == "wall" and s.outsideBoundaryCondition().lower() == "surface"
    ],
    key=lambda s: s.nameString(),
)
assert len(int_walls) == 4, f"Expected 2 interior walls (+ 2 reversed), found {len(int_walls)}"
int_wall1 = int_walls[0]
ss = add_door(int_wall1, "Door")
ss.setConstruction(c_opaque)

int_wall2 = next(w for w in int_walls[1:] if w.adjacentSurface().get() != int_wall1)
ss = add_window(int_wall2, subsurface_type="FixedWindow")
ss.setConstruction(c_window)

m.save("test.osm", True)

ft = openstudio.energyplus.ForwardTranslator()
ft.setExcludeHTMLOutputReport(True)
ft.setExcludeLCCObjects(True)
ft.setExcludeSpaceTranslation(True)
ft.setExcludeVariableDictionary(True)
ft.setExcludeSQliteOutputReport(True)
w = ft.translateModel(m)

# Cleanup pass

print({x.iddObject().name() for x in w.objects()})
obj_types_to_keep = {
    #'Building',
    "BuildingSurface:Detailed",
    "Construction",
    "FenestrationSurface:Detailed",
    # "GlobalGeometryRules",
    "Material",
    "Material:NoMass",
    #'OutdoorAir:Node',
    #'RunPeriod',
    #'Schedule:Constant',
    #'ScheduleTypeLimits',
    #'SimulationControl',
    #'Sizing:Parameters',
    #'Timestep',
    "WindowMaterial:SimpleGlazingSystem",
    "Zone",
}

for o in w.objects():
    if o.iddObject().name() not in obj_types_to_keep:
        o.remove()

w.save("test.idf", True)

[openstudio.energyplus.ForwardTranslator] <0> Both surfaces 'Space2 Wall 3', and 'Space1 Wall 1' reference the same construction 'Opaque Construction' but it is not symmetric, creating a reversed copy.
{'Sizing:Parameters', 'WindowMaterial:SimpleGlazingSystem', 'Schedule:Constant', 'Construction', 'Zone', 'Building', 'Timestep', 'Material', 'Material:NoMass', 'GlobalGeometryRules', 'SimulationControl', 'BuildingSurface:Detailed', 'FenestrationSurface:Detailed', 'RunPeriod', 'ScheduleTypeLimits', 'OutdoorAir:Node'}
[openstudio.energyplus.ForwardTranslator] <0> Both surfaces 'Space4 Wall 3', and 'Space3 Wall 1' reference the same construction 'Opaque Construction' but it is not symmetric, creating a reversed copy.
[openstudio.energyplus.ForwardTranslator] <0> Both surfaces 'Space1 RoofCeiling', and 'Space3 Floor' reference the same construction 'Opaque Construction' but it is not symmetric, creating a reversed copy.
[openstudio.energyplus.ForwardTranslator] <0> Both surfaces 'Space4 Floo

True

In [73]:
NEW_IDD_PATH = Path("/home/julien/Software/Others/EnergyPlus-build-release/Products/Energy+.idd")
idf_path = Path("test.idf")

In [74]:
idd = openstudio.IddFile.load(NEW_IDD_PATH).get()
w = openstudio.Workspace.load(idf_path, idd).get()

In [75]:
objects = get_constructions_and_materials()

In [76]:
w.addObjects(objects)

(<openstudio.openstudioutilitiesidf.WorkspaceObject; proxy of <Swig Object of type 'openstudio::WorkspaceObject *' at 0x787830195ef0> >,
 <openstudio.openstudioutilitiesidf.WorkspaceObject; proxy of <Swig Object of type 'openstudio::WorkspaceObject *' at 0x787830195a10> >,
 <openstudio.openstudioutilitiesidf.WorkspaceObject; proxy of <Swig Object of type 'openstudio::WorkspaceObject *' at 0x787830197d80> >,
 <openstudio.openstudioutilitiesidf.WorkspaceObject; proxy of <Swig Object of type 'openstudio::WorkspaceObject *' at 0x787830197600> >,
 <openstudio.openstudioutilitiesidf.WorkspaceObject; proxy of <Swig Object of type 'openstudio::WorkspaceObject *' at 0x787830196610> >,
 <openstudio.openstudioutilitiesidf.WorkspaceObject; proxy of <Swig Object of type 'openstudio::WorkspaceObject *' at 0x7878301961f0> >,
 <openstudio.openstudioutilitiesidf.WorkspaceObject; proxy of <Swig Object of type 'openstudio::WorkspaceObject *' at 0x787830196700> >,
 <openstudio.openstudioutilitiesidf.Works

In [80]:
surface_types = set()
surface_boundary_types = set()

for o in w.getObjectsByType(idd.getObject("BuildingSurface:Detailed").get()):
    # o.setString(2, "")
    surface_type = o.getString(1).get()
    surface_boundary_type = o.getString(5).get()
    surface_types.add(surface_type)
    surface_boundary_types.add(surface_boundary_type)

In [82]:
surface_boundary_types

{'Adiabatic', 'Ground', 'Outdoors', 'Surface'}

In [154]:
surface_infos = [
  {
    "name": "Space4 Wall 4",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space4 Wall 3",
    "construction": "Interior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space4 Wall 2",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space4 Wall 1",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space4 RoofCeiling",
    "construction": "Exterior Roof Construction",
    "surface_type": "Roof",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space4 Floor",
    "construction": "Interior Floor Construction",
    "surface_type": "Floor",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space3 Wall 4",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space3 Wall 3",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space3 Wall 2",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space3 Wall 1",
    "construction": "Interior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space3 RoofCeiling",
    "construction": "Exterior Roof Construction",
    "surface_type": "Roof",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space3 Floor",
    "construction": "Interior Floor Construction",
    "surface_type": "Floor",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space2 Wall 4",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space2 Wall 3",
    "construction": "Interior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space2 Wall 2",
    "construction": "Ground Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Ground"
  },
  {
    "name": "Space2 Wall 1",
    "construction": "Adiabatic Surface Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Adiabatic"
  },
  {
    "name": "Space2 RoofCeiling",
    "construction": "Interior Roof Construction",
    "surface_type": "Roof",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space2 Floor",
    "construction": "Ground Floor Construction",
    "surface_type": "Floor",
    "surface_boundary_type": "Ground"
  },
  {
    "name": "Space1 Wall 4",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space1 Wall 3",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space1 Wall 2",
    "construction": "Exterior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "Space1 Wall 1",
    "construction": "Interior Wall Construction",
    "surface_type": "Wall",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space1 RoofCeiling",
    "construction": "Interior Roof Construction",
    "surface_type": "Roof",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "Space1 Floor",
    "construction": "Exterior Floor Construction",
    "surface_type": "Floor",
    "surface_boundary_type": "Exterior"
  }
]
subsurface_infos = [
  {
    "name": "InteriorWindow - FixedWindow - Reversed",
    "construction": "Interior FixedWindow Construction",
    "surface_type": "FixedWindow",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "ExteriorWindow - Skylight",
    "construction": "Exterior Skylight Construction",
    "surface_type": "Skylight",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "ExteriorWindow - Window",
    "construction": "Exterior FixedWindow Construction",
    "surface_type": "Window",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "ExteriorWindow - OperableWindow",
    "construction": "Exterior OperableWindow Construction",
    "surface_type": "OperableWindow",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "InteriorWindow - FixedWindow",
    "construction": "Interior FixedWindow Construction",
    "surface_type": "FixedWindow",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "ExteriorWindow - FixedWindow",
    "construction": "Exterior FixedWindow Construction",
    "surface_type": "FixedWindow",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "InteriorDoor - Door - Reversed",
    "construction": "Interior Door Construction",
    "surface_type": "Door",
    "surface_boundary_type": "Interior"
  },
  {
    "name": "ExteriorDoor - GlassDoor",
    "construction": "Exterior GlassDoor Construction",
    "surface_type": "GlassDoor",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "ExteriorDoor - OverheadDoor",
    "construction": "Exterior OverheadDoor Construction",
    "surface_type": "OverheadDoor",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "ExteriorDoor - Door",
    "construction": "Exterior Door Construction",
    "surface_type": "Door",
    "surface_boundary_type": "Exterior"
  },
  {
    "name": "InteriorDoor - Door",
    "construction": "Interior Door Construction",
    "surface_type": "Door",
    "surface_boundary_type": "Interior"
  }
]



In [155]:
import pandas as pd


df = pd.DataFrame(surface_infos)

df.set_index(["surface_boundary_type", "surface_type"], inplace=True)
df.sort_index(axis=0, inplace=True)
surface_types = ["Floor", "Wall", "Roof"]
surface_boundary_types = ["Exterior", "Interior", "Ground", "Adiabatic"]

for surface_boundary_type in surface_boundary_types:
    print(f"    // {surface_boundary_type} Surfaces")
    df2 = df.loc[surface_boundary_type]
    for surface_type in [s for s in surface_types if s in df2.index]:
        print(f"    /// {surface_boundary_type} {surface_type}")
        df3 = df2.loc[[surface_type]]
        for _, s in df3.iterrows():
            sf_name = s["name"]
            cons_name = s["construction"]
            print(f'    checkDefaultConstruction("{cons_name}", "{sf_name}");')
        print("")
    print("\n")

    // Exterior Surfaces
    /// Exterior Floor
    checkDefaultConstruction("Exterior Floor Construction", "Space1 Floor");

    /// Exterior Wall
    checkDefaultConstruction("Exterior Wall Construction", "Space4 Wall 4");
    checkDefaultConstruction("Exterior Wall Construction", "Space4 Wall 2");
    checkDefaultConstruction("Exterior Wall Construction", "Space4 Wall 1");
    checkDefaultConstruction("Exterior Wall Construction", "Space3 Wall 4");
    checkDefaultConstruction("Exterior Wall Construction", "Space3 Wall 3");
    checkDefaultConstruction("Exterior Wall Construction", "Space3 Wall 2");
    checkDefaultConstruction("Exterior Wall Construction", "Space2 Wall 4");
    checkDefaultConstruction("Exterior Wall Construction", "Space1 Wall 4");
    checkDefaultConstruction("Exterior Wall Construction", "Space1 Wall 3");
    checkDefaultConstruction("Exterior Wall Construction", "Space1 Wall 2");

    /// Exterior Roof
    checkDefaultConstruction("Exterior Roof Construction", 

In [157]:
df = pd.DataFrame(subsurface_infos)

df.set_index(["surface_boundary_type", "surface_type"], inplace=True)
df.sort_index(axis=0, inplace=True)
subsurface_types = [
    "Window"
    "FixedWindow",
    "OperableWindow",
    "Door",
    "GlassDoor",
    "OverheadDoor",
    "Skylight",
    "TubularDaylightDome",
    "TubularDaylightDiffuser",
]

subsurface_boundary_types = ["Exterior", "Interior"]

for subsurface_boundary_type in subsurface_boundary_types:
    print(f"    // {subsurface_boundary_type} SubSurfaces")
    df2 = df.loc[subsurface_boundary_type]
    for subsurface_type in [s for s in subsurface_types if s in df2.index]:
        print(f"    /// {subsurface_boundary_type} {subsurface_type}")
        df3 = df2.loc[[subsurface_type]]
        for _, s in df3.iterrows():
            sf_name = s["name"]
            cons_name = s["construction"]
            print(f'    checkDefaultConstruction("{cons_name}", "{sf_name}");')
        print("")
    print("\n")

name  \
surface_boundary_type surface_type                                              
Exterior              Door                                ExteriorDoor - Door   
                      FixedWindow                ExteriorWindow - FixedWindow   
                      GlassDoor                      ExteriorDoor - GlassDoor   
                      OperableWindow          ExteriorWindow - OperableWindow   
                      OverheadDoor                ExteriorDoor - OverheadDoor   
                      Skylight                      ExteriorWindow - Skylight   
                      Window                          ExteriorWindow - Window   
Interior              Door                     InteriorDoor - Door - Reversed   
                      Door                                InteriorDoor - Door   
                      FixedWindow     InteriorWindow - FixedWindow - Reversed   
                      FixedWindow                InteriorWindow - FixedWindow   

                                                              construction  
surface_boundary_type surface_type                                          
Exterior              Door                      Exterior Door Construction  
                      FixedWindow        Exterior FixedWindow Construction  
                      GlassDoor            Exterior GlassDoor Construction  
                      OperableWindow  Exterior OperableWindow Construction  
                      OverheadDoor      Exterior OverheadDoor Construction  
                      Skylight              Exterior Skylight Construction  
                      Window             Exterior FixedWindow Construction  
Interior              Door                      Interior Door Construction  
                      Door                      Interior Door Construction  
                      FixedWindow        Interior FixedWindow Construction  
                      FixedWindow        Interior FixedWindow Construction

    // Exterior SubSurfaces
    /// Exterior FixedWindow
    checkDefaultConstruction("Exterior FixedWindow Construction", "ExteriorWindow - FixedWindow");

    /// Exterior OperableWindow
    checkDefaultConstruction("Exterior OperableWindow Construction", "ExteriorWindow - OperableWindow");

    /// Exterior Door
    checkDefaultConstruction("Exterior Door Construction", "ExteriorDoor - Door");

    /// Exterior GlassDoor
    checkDefaultConstruction("Exterior GlassDoor Construction", "ExteriorDoor - GlassDoor");

    /// Exterior OverheadDoor
    checkDefaultConstruction("Exterior OverheadDoor Construction", "ExteriorDoor - OverheadDoor");

    /// Exterior Skylight
    checkDefaultConstruction("Exterior Skylight Construction", "ExteriorWindow - Skylight");



    // Interior SubSurfaces
    /// Interior FixedWindow
    checkDefaultConstruction("Interior FixedWindow Construction", "InteriorWindow - FixedWindow - Reversed");
    checkDefaultConstruction("Interior FixedWindow Construc

In [141]:
df2 = df.loc["Adiabatic"]
"Wall" in df2.index

True

In [146]:
[s for s in surface_types if s in df2.index]

['Wall']

In [134]:
df3 = df2.loc[["Wall"]]
df3

,name,construction
surface_type,,
Wall,Space2 Wall 1,Adiabatic Surface Construction


In [137]:
for _, s in df3.iterrows():
    print(s["name"], s["construction"])

Space2 Wall 1 Adiabatic Surface Construction


In [100]:
for surface_boundary_type, df2 in df.groupby('surface_boundary_type'):
    print("   // {s

In [105]:
g = next(iter(grouped))
g

('Adiabatic',
              name                    construction surface_type  \
 15  Space2 Wall 1  Adiabatic Surface Construction         Wall   
 
    surface_boundary_type  
 15             Adiabatic  )

In [107]:
g[0],

('Adiabatic',
              name                    construction surface_type  \
 15  Space2 Wall 1  Adiabatic Surface Construction         Wall   
 
    surface_boundary_type  
 15             Adiabatic  )

In [108]:
g[1]

,name,construction,surface_type,surface_boundary_type
15,Space2 Wall 1,Adiabatic Surface Construction,Wall,Adiabatic


In [103]:
for g in grouped:
    print(g)

('Adiabatic',              name                    construction surface_type  \
15  Space2 Wall 1  Adiabatic Surface Construction         Wall   

   surface_boundary_type  
15             Adiabatic  )
('Exterior',                   name                construction surface_type  \
0        Space4 Wall 4  Exterior Wall Construction         Wall   
2        Space4 Wall 2  Exterior Wall Construction         Wall   
3        Space4 Wall 1  Exterior Wall Construction         Wall   
4   Space4 RoofCeiling  Exterior Roof Construction         Roof   
6        Space3 Wall 4  Exterior Wall Construction         Wall   
7        Space3 Wall 3  Exterior Wall Construction         Wall   
8        Space3 Wall 2  Exterior Wall Construction         Wall   
10  Space3 RoofCeiling  Exterior Roof Construction         Roof   
12       Space2 Wall 4  Exterior Wall Construction         Wall   
18       Space1 Wall 4  Exterior Wall Construction         Wall   
19       Space1 Wall 3  Exterior Wall Constructi

In [ ]:
for (key, ax) in zip(grouped.groups.keys(), axes.flatten()):
    grouped.get_group(key).plot(ax=ax)

In [96]:
grouped.groups.keys()

dict_keys([('Adiabatic', 'Wall'), ('Exterior', 'Roof'), ('Exterior', 'Wall'), ('Ground', 'Floor'), ('Ground', 'Wall'), ('Interior', 'Floor'), ('Interior', 'Roof'), ('Interior', 'Wall')])

In [77]:
w.save(idf_path, True)

True

In [68]:
o = w.getObjectsByType(idd.getObject("BuildingSurface:Detailed").get())[0]

In [71]:
print(o)

BuildingSurface:Detailed,
  Space4 Wall 4,                          !- Name
  Wall,                                   !- Surface Type
  ,                                       !- Construction Name
  Zone4,                                  !- Zone Name
  ,                                       !- Space Name
  Outdoors,                               !- Outside Boundary Condition
  ,                                       !- Outside Boundary Condition Object
  SunExposed,                             !- Sun Exposure
  WindExposed,                            !- Wind Exposure
  ,                                       !- View Factor to Ground
  ,                                       !- Number of Vertices
  0, 10, 3,                               !- X,Y,Z Vertex 1 {m}
  10, 10, 3,                              !- X,Y,Z Vertex 2 {m}
  10, 10, 0,                              !- X,Y,Z Vertex 3 {m}
  0, 10, 0;                               !- X,Y,Z Vertex 4 {m}




In [70]:
o.setString(2, "")

True

In [31]:
w.setStrictnessLevel(openstudio.StrictnessLevel("None"))

True

In [50]:
o = idf_file.getObjectsByType(idd.getObject("BuildingSurface:Detailed").get())[0]